<a href="https://colab.research.google.com/github/nashsparrow/pytorch/blob/main/3.pytorch_mnist_cnn_manual_hyperParameter_tuning_testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
%matplotlib inline

import time

In [ ]:
#converts MNIST Image files into Tensor
transform = transforms.ToTensor()


In [ ]:
#train data
train_data = datasets.MNIST(root='cnn_data', train=True, download=True, transform=transform)

100%|██████████| 9.91M/9.91M [00:00<00:00, 12.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 337kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.21MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.9MB/s]


In [ ]:
test_data = datasets.MNIST(root='cnn_data', train=False, download=True, transform=transform)

In [ ]:
#Model Class
class ConvolutionalNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=3, stride=1)
    self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=3, stride=1)
    self.fc1 = nn.Linear(in_features=16*5*5, out_features=120) #output of the convolution layers 16*5*5, 120 arbitarary
    self.fc2 = nn.Linear(in_features=120, out_features=84)
    self.fc3 = nn.Linear(in_features=84, out_features=10)

  def forward(self, X):
    X = F.relu(self.conv1(X))
    X = F.max_pool2d(X, 2, 2)
    X = F.relu(self.conv2(X))
    X = F.max_pool2d(X, 2, 2)
    X = X.view(-1, 16*5*5) #-1 so the batch sie can be varied
    X = F.relu(self.fc1(X))
    X = F.relu(self.fc2(X))
    X = self.fc3(X)
    return F.log_softmax(X, dim=1)


In [ ]:
#Instance of the model

torch.manual_seed(41)


In [ ]:
#Manual Tuning
#Type 1 Randomly select learning rate for lr while keeping the batch_size, optimizer fixed
#Type 2 Randomly select optimizer while keeping the learning rate, batch size fixed
#Type 3 Randomly select batch_size while keeping the learning rate, optimizer fixed

random_configs = [
    {"type": 1, "config": 1, "lr": 0.1, "batch_size": 10, "optimizer":"adam"},
    {"type": 1, "config": 2, "lr": 0.01, "batch_size": 10, "optimizer":"adam"},
    {"type": 1, "config": 3, "lr": 0.001, "batch_size": 10, "optimizer":"adam"},
    {"type": 2, "config": 1, "lr": 0.01, "batch_size": 10, "optimizer":"sgd"},
    {"type": 2, "config": 2, "lr": 0.01, "batch_size": 10, "optimizer":"adamw"},
    {"type": 2, "config": 3, "lr": 0.01, "batch_size": 10, "optimizer":"rmsprop"},
    {"type": 3, "config": 1, "lr": 0.01, "batch_size": 10, "optimizer":"adam"},
    {"type": 3, "config": 2, "lr": 0.01, "batch_size": 20, "optimizer":"adam"},
    {"type": 3, "config": 3, "lr": 0.01, "batch_size": 40, "optimizer":"adam"},
]

results = []
epochs = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for i, config in enumerate(random_configs):
  model = ConvolutionalNetwork().to(device)

  train_losses = []
  test_losses = []
  train_correct = []
  test_correct = []

  learning_rate = config["lr"]
  batch_size = config["batch_size"]

  criterion = nn.CrossEntropyLoss()

  match config["optimizer"]:
    case "adam":
      optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    case "adamw":
      optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    case "sgd":
      optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    case "rmsprop":
      optimizer = torch.optim.RMSprop(model.parameters(), lr=learning_rate)

  train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
  test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

  for j in range(epochs):
    trn_corr = 0
    tst_corr = 0
    for b, (X_train, y_train) in enumerate(train_loader):
      X_train, y_train = X_train.to(device), y_train.to(device)
      y_pred = model(X_train)
      loss = criterion(y_pred, y_train) #one loss value for entire batch #cross entropy loss, calculates each loss and averages them

      predicted = torch.max(y_pred.data, 1)[1]
      batch_correct = (predicted == y_train).sum()
      trn_corr += batch_correct

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      if b%600 == 0:
        print(f'type: {config["type"]} config: {config["config"]} epoch: {j} batch: {b} loss: {loss.item()}')

    train_losses.append(loss)
    train_correct.append(trn_corr)

    #Test after batch training is done for this epoch

    with torch.no_grad():
      for b, (X_test, y_test) in enumerate(test_loader):
        X_test, y_test = X_test.to(device), y_test.to(device)
        y_val = model(X_test)
        predicted = torch.max(y_val.data, 1)[1]
        tst_corr += (predicted == y_test).sum()

      loss = criterion(y_val, y_test)
      test_losses.append(loss)
      test_correct.append(tst_corr)

  results.append([config, train_losses, test_losses, train_correct, test_correct])

type: 1 config: 1 epoch: 0 batch: 0 loss: 2.323486566543579
type: 1 config: 1 epoch: 0 batch: 600 loss: 2.1853582859039307
type: 1 config: 1 epoch: 0 batch: 1200 loss: 2.414130687713623
type: 1 config: 1 epoch: 0 batch: 1800 loss: 2.4662554264068604
type: 1 config: 1 epoch: 0 batch: 2400 loss: 2.4721462726593018
type: 1 config: 1 epoch: 0 batch: 3000 loss: 2.4240641593933105
type: 1 config: 1 epoch: 0 batch: 3600 loss: 2.3392412662506104
type: 1 config: 1 epoch: 0 batch: 4200 loss: 2.235496997833252
type: 1 config: 1 epoch: 0 batch: 4800 loss: 2.2505643367767334
type: 1 config: 1 epoch: 0 batch: 5400 loss: 2.307893991470337
type: 1 config: 1 epoch: 1 batch: 0 loss: 2.311593770980835
type: 1 config: 1 epoch: 1 batch: 600 loss: 2.2682464122772217
type: 1 config: 1 epoch: 1 batch: 1200 loss: 2.3045969009399414
type: 1 config: 1 epoch: 1 batch: 1800 loss: 2.361009120941162
type: 1 config: 1 epoch: 1 batch: 2400 loss: 2.414472818374634
type: 1 config: 1 epoch: 1 batch: 3000 loss: 2.44114565

In [ ]:
results

NameError: name 'results' is not defined